# REBULEX — reranker ölçümü (Colab)

`rerank.py` embedding aramasının getirdiği ilk 100 kararı bir reranker'la yeniden sıralar ve ölçer.
Kod GitHub'dan (`metin-temizleme` dalı), veriler Drive'dan gelir.

**Başlamadan önce**
1. `Çalışma zamanı > Çalışma zamanı türünü değiştir` → **A100**.
2. Drive'da `REBULEX/rerank_veri.zip` ve `REBULEX/kararlar_temiz.parquet` olmalı.
3. Soldaki 🔑 *Secrets* bölümünde `HF_TOKEN` olmalı (notebook erişimi açık).
4. Gece bırakılacaksa: laptop şarjda, uyku modu kapalı, bu sekme açık.

**Akış:** 1–4. hücreler, 5 (bellek testi, ~3 dk), 6 (gece kuyruğu). Sabah 7 (sonuçlar).
Her ölçüm bittiği anda Drive'a (`REBULEX/rerank_sonuc/`) yazılır; kesilirse 1–4 ve 6 tekrar çalıştırılır,
bitmiş ölçümler atlanır.

## 1. Drive'ı bağla ve GPU'yu kontrol et

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ayarlar
`REBULEX_RERANK_BATCH`: aynı anda puanlanan çift sayısı. Tam metinde çiftler 8192 tokena kadar uzun; 5. hücredeki
bellek testi hata verirse 4'e düşür.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/REBULEX"
SONUC = f"{DRIVE_DIR}/rerank_sonuc"
LOCAL_DATA = "/content/data"
REPO = "https://github.com/ahmetege0/rebulex.git"
BRANCH = "metin-temizleme"

os.environ["REBULEX_DATA"] = LOCAL_DATA     # config.py veriyi buradan okur
os.environ["REBULEX_RERANK_BATCH"] = "8"
os.environ["REBULEX_DTYPE"] = "bfloat16"    # sorgu vektörlerini çıkaran embedding modelleri için

## 3. Kodu GitHub'dan çek ve kur
Kütüphane sürümleri bilgisayardakiyle aynı tutulur (sentence-transformers 6.0.1, transformers 5.12.1).

In [ ]:
if not os.path.exists("/content/rebulex"):
    !git clone -q -b {BRANCH} {REPO} /content/rebulex
else:
    !cd /content/rebulex && git fetch -q && git reset -q --hard origin/{BRANCH}
%cd /content/rebulex
!git log --oneline -1
!pip install -q -r requirements.txt sentence-transformers==6.0.1 transformers==5.12.1
import sentence_transformers, transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN yüklendi")
except Exception as e:
    print("HF_TOKEN yok; EmbeddingGemma adayları çalışmaz:", type(e).__name__)

## 4. Verileri hazırla
- Zip Colab diskine açılır; Drive'daki önceki sonuçların `experiments.csv`'si geri yüklenir (pilot satırları kaybolmasın).
- Temiz havuz Drive'dan kopyalanır.
- Tam metin sonuç klasörleri doğrudan Drive'a bağlanır: her ölçüm bittiği anda Drive'da olur.

In [ ]:
import shutil

os.makedirs(SONUC, exist_ok=True)
if not os.path.exists(f"{LOCAL_DATA}/experiments.csv"):   # oturum içinde eklenen satırlar ezilmesin
    !unzip -q -o {DRIVE_DIR}/rerank_veri.zip -d {LOCAL_DATA}
    if os.path.exists(f"{SONUC}/experiments.csv"):
        shutil.copy(f"{SONUC}/experiments.csv", f"{LOCAL_DATA}/experiments.csv")
        print("Önceki experiments.csv Drive'dan geri yüklendi")
shutil.copy(f"{DRIVE_DIR}/kararlar_temiz.parquet", f"{LOCAL_DATA}/n2000/kararlar_temiz.parquet")

for folder in ["rerank_temiz", "rerank_orijinal"]:
    os.makedirs(f"{SONUC}/{folder}", exist_ok=True)
    if not os.path.exists(f"{LOCAL_DATA}/n2000/{folder}"):
        os.symlink(f"{SONUC}/{folder}", f"{LOCAL_DATA}/n2000/{folder}")
!find {LOCAL_DATA} -maxdepth 2 | sort

## 5. Bellek testi (~3 dk)
Her reranker'a bir grup en uzun (8192 tokenlık) çift verilir. Üçü de "sorunsuz" yazmalı.
Bellek hatası olursa 2. hücrede `REBULEX_RERANK_BATCH`'i 4 yap, 2. ve 5. hücreyi tekrar çalıştır.

In [ ]:
!python -u rerank.py --bellek-testi

## 6. Gece kuyruğu
Sıra: temiz metinle ModernBERT, bge, Qwen3; sonra orijinal metinle üçü. Her adım ayrı komut: biri hata verirse
sonraki adım yine çalışır. Her adımdan sonra `experiments.csv` Drive'a kopyalanır. Çıktı `rerank_sonuc/gece.log`'a da
yazılır.

`KAPAT = True`: kuyruk bitince çalışma zamanı kapanır, boşta birim harcanmaz.

In [ ]:
from datetime import datetime
from google.colab import runtime

KAPAT = True
steps = [
    "--metin temiz --reranker yok modernbert_tr",
    "--metin temiz --reranker bge_m3",
    "--metin temiz --reranker qwen3_06b",
    "--metin orijinal --reranker yok modernbert_tr",
    "--metin orijinal --reranker bge_m3",
    "--metin orijinal --reranker qwen3_06b",
]
for step in steps:
    print(f"\n##### {datetime.now():%H:%M} rerank.py {step}", flush=True)
    !python -u rerank.py {step} 2>&1 | tee -a {SONUC}/gece.log
    shutil.copy(f"{LOCAL_DATA}/experiments.csv", f"{SONUC}/experiments.csv")
print(f"Kuyruk bitti: {datetime.now():%H:%M}")
if KAPAT:
    runtime.unassign()

## 7. Sonuçlar (geniş ölçü)
Sabah: yeni bir oturumda 1–4. hücreleri çalıştırıp bunu çalıştırabilirsin (tablo Drive'daki dosyadan okunur).

In [ ]:
import pandas as pd

exp = pd.read_csv(f"{SONUC}/experiments.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
rows = exp[exp["not"].str.startswith("rerank")]
cols = ["not", "deneme", "test_seti", "reranker", "ilk1_genis_%", "ilk5_genis_%", "ilk10_genis_%",
        "ilk20_genis_%", "ilk100_genis_%", "mrr_genis", "rerank_sn"]
pd.set_option("display.max_colwidth", 80)
rows[cols]